# MSCS634 Deliverable 4: Final Insights, Recommendations, and Presentation
This notebook consolidates the full Diamonds data mining project across preprocessing, EDA, feature engineering, regression, classification, clustering, and association rule mining.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid')


## 1. Load Dataset
The Diamonds dataset contains 53,940 rows and 10 columns. It includes numerical and categorical variables that support the full data mining workflow.


In [ ]:
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()


## 2. Data Cleaning and EDA
The original work found no missing values and removed 146 duplicate rows. Price is strongly right-skewed, and carat has the strongest relationship with price.


In [ ]:
print(df.isna().sum())
print('Duplicate rows before:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplicate rows after:', df.duplicated().sum())

fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(df['price'], bins=50, kde=True, ax=ax)
ax.set_title('Diamond Price Distribution')
plt.show()

fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=df.sample(3000, random_state=42), x='carat', y='price', alpha=0.4, ax=ax)
ax.set_title('Carat vs Price')
plt.show()


## 3. Feature Engineering
Created volume, log transformations, encoded categorical variables, and a price-tier target for classification.


In [ ]:
df = df[(df['x'] > 0) & (df['y'] > 0) & (df['z'] > 0)].copy()
df['volume'] = df['x'] * df['y'] * df['z']
df['log_price'] = np.log1p(df['price'])
df['log_carat'] = np.log1p(df['carat'])
df['log_volume'] = np.log1p(df['volume'])
df['price_tier'] = pd.cut(df['price'], bins=[0,1000,5000,np.inf], labels=['Low','Medium','High'])
df[['price','carat','volume','log_price','price_tier']].head()


## 4. Regression Modeling
Regression predicts diamond price. In the previous deliverable, engineered features improved interpretability and performance. Carat/volume and quality grades were the most important predictors.


In [ ]:
features = ['carat','depth','table','x','y','z','cut','color','clarity']
X = df[features]
y = df['price']
num = ['carat','depth','table','x','y','z']
cat = ['cut','color','clarity']
pre = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
models = {'Linear Regression': LinearRegression(), 'Ridge': Ridge(alpha=1.0), 'Lasso': Lasso(alpha=0.001)}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
for name, model in models.items():
    pipe = Pipeline([('preprocess', pre), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(name, 'RMSE:', mean_squared_error(y_test, pred, squared=False), 'R2:', r2_score(y_test, pred))


## 5. Classification Modeling
The classification task predicts Low, Medium, or High price tier. Decision Tree and KNN performed very strongly, with the tuned Decision Tree reaching about 98.9% weighted F1 in Deliverable 3.


In [ ]:
X = df[['log_carat','log_volume','depth','table','cut','color','clarity']]
y = df['price_tier']
num = ['log_carat','log_volume','depth','table']
cat = ['cut','color','clarity']
pre = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
classifiers = {'Decision Tree': DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42), 'KNN': KNeighborsClassifier(n_neighbors=7)}
for name, clf in classifiers.items():
    pipe = Pipeline([('preprocess', pre), ('model', clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(name)
    print('Accuracy:', accuracy_score(y_test, pred))
    print('Weighted F1:', f1_score(y_test, pred, average='weighted'))
    print(classification_report(y_test, pred))


## 6. Clustering
K-Means was used to segment diamonds based on engineered size and quality variables. The elbow plot supported k=4 as a practical segmentation choice.


In [ ]:
cluster_df = df[['log_carat','log_volume','cut','color','clarity']].copy()
cluster_encoded = pd.get_dummies(cluster_df, drop_first=True)
scaled = StandardScaler().fit_transform(cluster_encoded)
wcss = []
for k in range(1,9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled)
    wcss.append(km.inertia_)
plt.figure(figsize=(8,5))
plt.plot(range(1,9), wcss, marker='o')
plt.title('K-Means Elbow Plot')
plt.xlabel('k')
plt.ylabel('WCSS')
plt.show()


## 7. Association Rule Mining
FP-Growth was used in Deliverable 3 to mine combinations of cut, color, clarity, and price tier. The most useful rules showed that high clarity and top color grades tend to associate with the High price tier, while lower clarity and mid-range color grades often associate with Low or Medium tiers.


## 8. Final Recommendations
- Use carat and derived volume as primary price predictors, but interpret them together with clarity and color.
- Use price-tier classification for inventory triage and customer search filters.
- Validate models externally before production use because historical market data may contain pricing bias.
- Add external variables such as certification lab, market date, and seller type for future improvement.
